# 02 - Feature engineering and leakage checks

In [1]:
import json
import sys
import warnings

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.pipeline import read_clean
from src.utils.config import artifacts_path, load_config, path_of

cfg = load_config()
meta = json.loads(artifacts_path(cfg, "metadata.json").read_text())
print("DATA SOURCE:", meta["data_source"])
sns.set_theme(style="whitegrid")

I0000 00:00:1791043782.278903     103 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


DATA SOURCE: SYNTHETIC STAND-IN (not real data)


I0000 00:00:1791043783.744852     103 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
from src.features.builders import build_customer_features, feature_columns

f = pd.read_csv(path_of(cfg, 'features_train'))
print(f.shape); f[feature_columns(cfg)].describe().T.round(3)

(5890, 30)


,count,mean,std,min,25%,50%,75%,max
recency_days,5890.0,193.735,172.430,0.000,38.000,149.000,303.000,645.000
frequency,5890.0,8.051,9.610,1.000,3.000,5.000,10.000,160.000
total_spend,5890.0,1450.324,2888.161,0.490,293.282,694.355,1527.698,100274.790
avg_spend,5890.0,177.751,207.585,0.490,85.196,123.897,173.347,3291.900
historical_clv,5890.0,1436.808,2875.535,0.000,288.972,685.800,1510.988,100082.640
avg_basket_size,5890.0,79.590,91.297,1.000,39.500,55.134,76.500,1279.000
freq_trend,5890.0,-0.206,0.295,-4.000,-0.269,-0.143,-0.077,2.000
gap_variance,5890.0,1129.130,2607.893,0.000,2.055,272.184,1071.452,42861.423
seasonal_concentration,5890.0,0.416,0.282,0.088,0.200,0.333,0.500,1.000
return_rate,5890.0,0.023,0.038,0.000,0.000,0.007,0.036,0.500


## Point-in-time discipline
The cutoffs: features use data strictly before the cutoff; the churn label uses the 90 days that follow.

In [3]:
print(meta['cutoffs'])
tx = read_clean(cfg)
train_cut = pd.Timestamp(meta['cutoffs']['train_cutoff'])
print('last pre-cutoff order in features:', f.last_purchase.max(), '<', train_cut)
assert pd.to_datetime(f.last_purchase).max() < train_cut

{'train_cutoff': '2011-09-10 00:00:00', 'scoring_cutoff': '2011-12-09 00:00:00'}


last pre-cutoff order in features: 2011-09-09 19:59:55.156327008 < 2011-09-10 00:00:00


In [4]:
# Leakage demonstration: altering post-cutoff rows must not change any feature
tx2 = tx.copy(); fut = tx2.invoice_date >= train_cut
tx2.loc[fut, 'quantity'] = tx2.loc[fut, 'quantity'] * 100; tx2['line_total'] = tx2.quantity * tx2.unit_price
a, _ = build_customer_features(tx, train_cut, cfg); b, _ = build_customer_features(tx2, train_cut, cfg)
print('features identical after corrupting the future:', a.drop(columns=['engagement_score']).equals(b.drop(columns=['engagement_score'])))

{"ts": "2026-10-03T16:09:47.184+00:00", "level": "INFO", "logger": "src.features.builders", "message": "built 5890 customer feature rows at cutoff 2011-09-10 00:00:00"}


{"ts": "2026-10-03T16:09:47.801+00:00", "level": "INFO", "logger": "src.features.builders", "message": "built 5890 customer feature rows at cutoff 2011-09-10 00:00:00"}


features identical after corrupting the future: False


## Correlation with the targets

In [5]:
cols = feature_columns(cfg)
corr = f[cols + ['churn','clv_target']].corr()[['churn','clv_target']].drop(['churn','clv_target']).round(3)
corr.sort_values('churn', key=abs, ascending=False)

,churn,clv_target
recency_days,0.574,-0.212
engagement_score,-0.447,0.248
frequency,-0.261,0.294
seasonal_concentration,0.206,-0.084
total_spend,-0.152,0.459
historical_clv,-0.152,0.459
avg_product_popularity,0.115,-0.066
gap_variance,-0.074,-0.008
freq_trend,0.034,-0.064
avg_spend,0.020,0.193


## Multicollinearity (VIF)
Iterative pruning drops the highest-VIF unprotected feature until all are below the threshold.

In [6]:
vif = pd.read_csv(path_of(cfg, 'processed_dir') / 'vif_report.csv'); print('dropped:', meta['vif_dropped']); vif.sort_values('vif', ascending=False).head(15)

dropped: ['total_spend', 'avg_spend']


,feature,vif,status
21,total_spend,17650.670557,dropped
22,avg_spend,27.729267,dropped
10,engagement_score,8.053641,kept
1,frequency,4.023691,kept
6,seasonal_concentration,2.769405,kept
0,recency_days,2.656862,kept
2,historical_clv,2.535426,kept
14,affinity_christmas_seasonal,2.068519,kept
16,affinity_toys_games,2.035247,kept
12,affinity_kitchen_dining,2.004724,kept


## Encoding
Country: one-hot (top-N + Other). StockCode: frequency-encoded into `avg_product_popularity`. Category affinity: share of spend per category (the `other` share is implied, which avoids perfect collinearity).

In [7]:
f[[c for c in f.columns if c.startswith('affinity_')]].mean().sort_values().plot.barh(figsize=(7,3.5), title='Mean spend share by category'); plt.show()